# FROST Key Generation

This notebook explains the **key generation phase** used for FROST-style threshold signing.

This module builds directly on two earlier notebooks:

- [01-threshold-polynomials.ipynb](01-threshold-polynomials.ipynb), which explains Shamir secret sharing, interpolation, and Feldman-style share verification
- [02-schnorr.ipynb](02-schnorr.ipynb), which explains the Schnorr-style proof structure reused in KeyGen

The goal of key generation is to produce:

- a long-lived secret signing share $s_i$ for each participant $P_i$
- a public verification share $Y_i = g^{s_i}$ for each participant
- one group public key $Y$ for the whole threshold signing group

No participant should learn the full secret key, but the group should still end up with a public key that can verify threshold-produced signatures.

## What KeyGen Is Doing

Each participant acts like a small dealer for their own random polynomial, and the group combines all of those polynomials into one joint secret polynomial.

This is the same polynomial-sharing idea introduced in [01-threshold-polynomials.ipynb](01-threshold-polynomials.ipynb), except now every participant contributes one polynomial rather than relying on a single dealer.

If participant $P_i$ creates polynomial:

$$
f_i(x) = \sum_{j=0}^{t-1} a_{ij} x^j,
$$

then the group's effective secret polynomial is:

$$
F(x) = \sum_{i=1}^{n} f_i(x).
$$

Each participant eventually holds one point on this combined polynomial:

$$
s_i = F(i).
$$

The hidden group secret is the constant term:

$$
s = F(0).
$$

That secret is never reconstructed during key generation.

## What DKG Actually Is (for FROST KeyGen)

**Distributed Key Generation (DKG)** is a protocol for creating one shared signing
key without ever constructing the full secret key on any single machine.

Instead of a trusted dealer choosing one secret and splitting it, every participant
contributes randomness, and the group secret emerges from combining all participant
contributions.

In this notebook's KeyGen flow, DKG works as follows:

1. Each participant picks random polynomial coefficients and defines their own
   polynomial $f_i(x)$.

2. They publish commitments to those coefficients so others can verify consistency
   later.

3. They privately send each participant a share value from their polynomial.

4. Each receiver verifies every incoming share against the sender's commitments.

5. Each participant sums all valid incoming shares to get one long-lived secret
   share $s_i$.

So DKG in FROST KeyGen produces:

- private share for participant $i$:

$$
s_i = \sum_{\ell=1}^{n} f_{\ell}(i)
$$

- participant's public verification share:

$$
Y_i = g^{s_i}
$$

- group public key:

$$
Y = \prod_{j=1}^{n} \phi_{j0} = g^{F(0)}
$$

where $F(x) = \sum_i f_i(x)$ is the combined secret polynomial.

The key point: no participant learns $F(0)$ directly, but everyone gets a valid
share tied to the same hidden group secret.

This is exactly why DKG is the foundation of threshold key ownership in FROST:
shared key creation without a trusted dealer and without exposing the full private
key at setup time.

## Round 1: Commit to Polynomials

Each participant $P_i$ performs the following steps.

### 1. Sample coefficients and define a polynomial

Participant $P_i$ samples $t$ random values:

$$
(a_{i0}, a_{i1}, \dots, a_{i(t-1)}) \xleftarrow{\$} \mathbb{Z}_q
$$

and defines a degree-$(t-1)$ polynomial:

$$
f_i(x) = \sum_{j=0}^{t-1} a_{ij} x^j.
$$

The constant term $a_{i0}$ is the participant's contribution to the eventual group secret.

### 2. Prove knowledge of the constant term

Participant $P_i$ proves knowledge of $a_{i0}$ by producing a Schnorr-style proof:

$$
\sigma_i = (R_i, \mu_i),
$$

where:

$$
k \xleftarrow{\$} \mathbb{Z}_q, \qquad R_i = g^k,
$$

$$
c_i = H(i, \Phi, g^{a_{i0}}, R_i),
$$

$$
\mu_i = k + a_{i0} \cdot c_i.
$$

This is the same proof pattern described in [02-schnorr.ipynb](02-schnorr.ipynb): commit with randomness, hash to a challenge, then respond with a value that proves knowledge of the secret exponent.

Here $\Phi$ is a context string used to prevent replay across different sessions.

### 3. Publish polynomial commitments

Participant $P_i$ computes a commitment vector:

$$
\tilde{C}_i = (\phi_{i0}, \phi_{i1}, \dots, \phi_{i(t-1)}),
$$

with

$$
\phi_{ij} = g^{a_{ij}}.
$$

This lets other participants later verify whether received shares are consistent with the committed polynomial.

### 4. Broadcast commitments and proof

Each participant broadcasts:

$$
(\tilde{C}_i, \sigma_i).
$$

### 5. Verify everyone else's proof

For every other participant $P_{\ell}$, participant $P_i$ verifies the proof

$$
\sigma_{\ell} = (R_{\ell}, \mu_{\ell})
$$

by checking:

$$
R_{\ell} \stackrel{?}{=} g^{\mu_{\ell}} \cdot \phi_{\ell 0}^{-c_{\ell}},
$$

where

$$
c_{\ell} = H(\ell, \Phi, \phi_{\ell 0}, R_{\ell}).
$$

If this fails, the protocol aborts. If it succeeds, the temporary proofs can be deleted.

## Round 2: Exchange and Verify Shares

After commitments are accepted, participants exchange actual secret shares.

### 1. Send shares privately

Each participant $P_i$ sends each participant $P_{\ell}$ the value:

$$
(\ell, f_i(\ell)).
$$

Participant $P_i$ keeps its own local share $(i, f_i(i))$.

### 2. Verify received shares

If participant $P_i$ receives share $f_{\ell}(i)$ from participant $P_{\ell}$, they check that it matches the previously published commitment vector:

$$
g^{f_{\ell}(i)} \stackrel{?}{=} \prod_{k=0}^{t-1} \phi_{\ell k}^{i^k}.
$$

If the equality fails, the share is invalid and the protocol aborts.

This is exactly the Feldman-style consistency check introduced in [01-threshold-polynomials.ipynb](01-threshold-polynomials.ipynb): the private share must agree with the committed polynomial coefficients.

### 3. Compute long-lived private signing share

After all shares are verified, participant $P_i$ sums the shares they received:

$$
s_i = \sum_{\ell=1}^{n} f_{\ell}(i).
$$

This $s_i$ is the participant's long-lived secret signing share.

### 4. Compute public verification shares and group key

Each participant computes their own public verification share:

$$
Y_i = g^{s_i}.
$$

The group public key is:

$$
Y = \prod_{j=1}^{n} \phi_{j0}.
$$

Since each $\phi_{j0} = g^{a_{j0}}$, this means:

$$
Y = g^{\sum_{j=1}^{n} a_{j0}} = g^{F(0)} = g^s.
$$

So the group public key corresponds exactly to the hidden group secret, even though no participant ever learns that secret directly.

## What This Achieves

At the end of KeyGen:

- each participant has one private share $s_i$
- each participant can publish or use a public verification share $Y_i$
- the group has one public key $Y$
- the full secret key exists only implicitly as the hidden value $F(0)$

This is what makes threshold signing possible: later, participants use their shares $s_i$ to produce partial signing responses that combine into a standard Schnorr signature under the group public key $Y$.

## Why the Proof and Commitments Matter

The proof of knowledge in Round 1 prevents a participant from publishing a constant-term commitment without actually knowing the corresponding discrete logarithm.

The commitment vector ensures that every secret share sent in Round 2 can be checked against a public polynomial commitment.

Together, these mechanisms prevent malformed inputs from silently corrupting the threshold key setup.

This is why FROST KeyGen is more than just “everyone picks a random share.” It is a distributed protocol for building one joint secret polynomial in a verifiable way.